## 1. Persiapan library dan konfigurasi

In [ ]:
# BLOK 1 — Memuat library dan menetapkan parameter agar eksperimen dapat direproduksi.
# Jika diperlukan di Google Colab:
# !pip -q install openpyxl

import re
import time
import warnings
from itertools import permutations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Konfigurasi utama yang boleh diubah
K_MIN, K_MAX = 2, 3
K_FINAL = 3  # penyederhanaan menjadi tiga topik utama
MAX_FEATURES = 1500
MIN_DF = 2
MAX_DF = 0.95
N_LSA_COMPONENTS = 25
N_PARTICLES = 30
N_ITERATIONS = 40
EXPERIMENT_SEEDS = list(range(10))
INERTIA_WEIGHT = 0.72
COGNITIVE_WEIGHT = 1.49
SOCIAL_WEIGHT = 1.49

print('Konfigurasi siap.')

## 2. Memuat dua dataset

In [ ]:
# BLOK 2 — Mencari dan memuat data ulasan bersih serta data mentah dari lokasi yang tersedia.
def temukan_file(nama_file):
    kandidat = [
        Path(nama_file),
        Path('upload') / nama_file,
        Path('/content') / nama_file,
    ]
    for path in kandidat:
        if path.exists():
            return path
    raise FileNotFoundError(
        f'{nama_file} tidak ditemukan. Letakkan file di folder yang sama dengan notebook '
        'atau unggah ke /content jika menggunakan Google Colab.'
    )

clean_path = temukan_file('dataset_review_clean_final.xlsx')
raw_path = temukan_file('seluruh_review_Bapenda_Sulsel_Mobile.xlsx')

df_clean = pd.read_excel(clean_path, sheet_name='Sheet1')
df_raw = pd.read_excel(raw_path, sheet_name='Review')

assert 'review_clean_final' in df_clean.columns, 'Kolom review_clean_final tidak ditemukan.'
assert 'content' in df_raw.columns, 'Kolom content tidak ditemukan pada data mentah.'

print(f'Data bersih : {df_clean.shape[0]:,} baris × {df_clean.shape[1]} kolom')
print(f'Data mentah : {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom')
display(df_clean.head())
display(df_raw[['content', 'score', 'at', 'appVersion']].head())

## 3. Validasi dan ringkasan data

In [ ]:
# BLOK 3 — Memeriksa kelayakan data, menghapus data tidak valid, lalu menampilkan ringkasannya.
# Validasi memastikan hanya ulasan teks yang layak dipakai model.
n_awal = len(df_clean)
data = df_clean[['review_clean_final']].copy()
data['review_clean_final'] = data['review_clean_final'].astype(str).str.strip()
data = data[data['review_clean_final'].str.len() > 0]
data = data.drop_duplicates(subset='review_clean_final').reset_index(drop=True)

ringkasan = pd.DataFrame({
    'indikator': [
        'Jumlah ulasan bersih awal',
        'Jumlah ulasan setelah validasi',
        'Duplikat/kosong terhapus',
        'Jumlah ulasan mentah',
        'Rata-rata skor data mentah',
    ],
    'nilai': [n_awal, len(data), n_awal - len(data), len(df_raw), round(df_raw['score'].mean(), 3)]
})
display(ringkasan)

score_counts = df_raw['score'].value_counts().sort_index()
ax = score_counts.plot(kind='bar', color='#276FBF', figsize=(7, 4))
ax.set_title('Distribusi Skor Ulasan pada Data Mentah')
ax.set_xlabel('Skor')
ax.set_ylabel('Jumlah ulasan')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Normalisasi, TF-IDF, dan Latent Semantic Analysis (LSA)

In [ ]:
# BLOK 4 — Menyiapkan teks menjadi vektor numerik TF-IDF dan mereduksi dimensinya dengan LSA.
STOPWORDS_ID = {
    'ada','adalah','agar','akan','aku','anda','atau','bagi','bahwa','bisa','buat','dalam','dan',
    'dari','dengan','di','dia','ini','itu','jadi','juga','karena','ke','kok','lagi','lebih','maka',
    'masih','mau','mohon','nya','oleh','pada','para','pun','saja','sangat','saya','sebagai','seperti',
    'sudah','supaya','tak','tidak','untuk','yang','ya','yg','aplikasi','app','bapenda','sulsel'
}
NORMALISASI_KATA = {
    'tdk':'tidak', 'gk':'tidak', 'ga':'tidak', 'gak':'tidak', 'nggak':'tidak',
    'sy':'saya', 'sya':'saya', 'knpa':'kenapa', 'klo':'kalau', 'kalo':'kalau',
    'sdh':'sudah', 'blm':'belum', 'dgn':'dengan', 'utk':'untuk', 'krn':'karena',
    'tp':'tetapi', 'tpi':'tetapi', 'jd':'jadi', 'org':'orang', 'apk':'aplikasi',
    'aplikasinya':'aplikasi', 'daplikasi':'di aplikasi', 'disamsat':'di samsat',
    'pajakada':'pajak ada', 'carax':'caranya', 'biayany':'biayanya'
}

# Normalisasi menyatukan variasi penulisan agar kata dengan makna sama dihitung sebagai fitur yang sama.
def normalisasi_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r'http\S+|www\.\S+', ' ', teks)
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    token_baru = []
    for token in teks.split():
        token_baru.extend(NORMALISASI_KATA.get(token, token).split())
    return re.sub(r'\s+', ' ', ' '.join(token_baru)).strip()

# Kolom asli dipertahankan; kolom baru ini khusus digunakan sebagai masukan model.
data['teks_model'] = data['review_clean_final'].map(normalisasi_teks)
data = data[data['teks_model'].str.len() > 0].reset_index(drop=True)

# TF-IDF memberi bobot tinggi pada kata/frasa yang khas; unigram dan bigram menangkap konteks pendek.
vectorizer = TfidfVectorizer(
    stop_words=list(STOPWORDS_ID), ngram_range=(1, 2), min_df=MIN_DF, max_df=MAX_DF,
    max_features=MAX_FEATURES, sublinear_tf=True
)
# Normalisasi L2 menyetarakan panjang vektor sehingga ulasan panjang tidak otomatis lebih dominan.
X_tfidf = normalize(vectorizer.fit_transform(data['teks_model']), norm='l2', copy=False)
feature_names = np.asarray(vectorizer.get_feature_names_out())

# Jumlah komponen dibatasi ukuran data agar TruncatedSVD selalu menerima dimensi yang valid.
n_components = min(N_LSA_COMPONENTS, X_tfidf.shape[1] - 1, X_tfidf.shape[0] - 1)
lsa = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
X = normalize(lsa.fit_transform(X_tfidf), norm='l2')

print('Matriks TF-IDF:', X_tfidf.shape)
print('Sparsity TF-IDF: {:.2%}'.format(1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])))
print('Matriks LSA untuk klastering:', X.shape)
print('Variansi kumulatif LSA: {:.2%}'.format(lsa.explained_variance_ratio_.sum()))
assert X.shape[0] >= K_MAX, 'Jumlah dokumen terlalu sedikit.'
assert X.shape[1] > 1, 'Dimensi LSA tidak mencukupi.'